# SETUP PACKAGES

In [139]:
%pip install google-genai
%pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# DO NOT CHANGE ANYTHING IN THE CELL BELOW

In [140]:
import os
from google import genai
from google.genai import types
from google import genai
from pydantic import BaseModel, Field
from typing import List
from copy import deepcopy

os.environ["GOOGLE_CLOUD_PROJECT"] = "project-038ccd57-3d62-4aac-8b5"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

client = genai.Client()

# ABLE TO CHANGE

In [141]:
import json

model_LLM = "gemini-2.5-flash"

PUZZLE_FILE = "dataset/puzzle_06.json"

with open(PUZZLE_FILE, 'r', encoding='utf-8') as f:
    puzzle = json.load(f)

# SETUP

## Convert cages to text format

In [142]:
cages_text = []
for cage in puzzle['cages']:
    cells = cage['cells']
    cage_sum = cage['sum']
    
    # Convert cell coordinates to text format (1-indexed)
    cell_strs = [f"r{r+1}c{c+1}" for r, c in cells]
    cage_str = "- " + " + ".join(cell_strs) + f" = {cage_sum}"
    cages_text.append(cage_str)

# Print the result
for cage_text in cages_text:
    print(cage_text)

- r1c1 + r1c2 = 6
- r1c3 + r1c4 = 16
- r1c5 + r2c5 = 7
- r1c6 + r1c7 = 9
- r1c8 + r1c9 = 9
- r2c1 + r3c1 + r3c2 + r4c2 = 26
- r4c1 + r5c1 = 4
- r2c2 + r2c3 = 6
- r2c4 + r2c6 + r3c4 + r3c5 + r3c6 = 21
- r2c7 + r2c8 = 13
- r2c9 + r3c8 + r3c9 + r4c8 = 21
- r3c3 + r4c3 + r4c4 = 11
- r4c5 + r5c4 + r5c5 + r5c6 = 21
- r3c7 + r4c6 + r4c7 = 21
- r5c2 + r5c3 = 10
- r5c7 + r5c8 = 10
- r4c9 + r5c9 = 14
- r6c1 + r6c2 = 12
- r6c3 + r6c4 + r7c3 = 17
- r7c1 + r7c2 + r8c1 + r8c2 + r9c1 = 27
- r9c2 + r9c3 = 7
- r7c4 + r8c3 + r8c4 = 17
- r6c5 + r7c5 + r8c5 = 12
- r6c6 + r6c7 + r7c7 = 16
- r6c8 + r6c9 = 7
- r7c6 + r8c6 + r8c7 = 21
- r9c4 + r9c5 + r9c6 = 16
- r9c7 + r9c8 = 7
- r7c8 + r7c9 + r8c8 + r8c9 + r9c9 = 21


## Convert puzzle grid to markdown table and back

In [143]:
def grid_to_markdown(grid):
    """Convert a 2D list grid to a markdown table string"""
    rows = []
    # Header row
    rows.append("| " + " | ".join([""] * len(grid[0])) + " |")
    # Separator row
    rows.append("|" + "|".join(["---"] * len(grid[0])) + "|")
    # Data rows
    for row in grid:
        rows.append("| " + " | ".join(str(cell) for cell in row) + " |")
    return "\n".join(rows)

def markdown_to_grid(markdown_str):
    """Convert a markdown table string back to a 2D list grid"""
    lines = [line.strip() for line in markdown_str.split("\n") if line.strip()]
    grid = []
    for line in lines[2:]:  # Skip header and separator rows
        # Remove leading and trailing pipes and split by pipe
        cells = [cell.strip() for cell in line.split("|")[1:-1]]
        grid.append([int(cell) for cell in cells if cell])
    return grid

markdown_table = grid_to_markdown(puzzle['puzzle'])
print(markdown_table)

|  |  |  |  |  |  |  |  |  |
|---|---|---|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |


## Load cheat-sheet

In [144]:
with open('killer_sudoku_cheat_sheet.md', 'r', encoding='utf-8') as file:
    cheat_sheet = file.read()

# Single Prompt

In [145]:
prompt = """
You are an expert puzzle solver. Your task is to solve a 6x6 Killer Sudoku grid based on the rules, strategies, and puzzle data provided below.

1. The Grid & Core Rules:
- The grid is 6x6, containing 6 rows and 6 columns.
- Notation: "r1c1" indicates the cell at row 1 and column 1.
- The grid consists of 6 rectangular 2x3 subgrids (2 rows tall, 3 columns wide).
- Standard Sudoku Rule: Each row, each column, and each 2x3 subgrid must contain the numbers 1 through 6 exactly once.
- Killer Sudoku Rule: The grid is divided into "Cages" (contiguous groups of cells). The sum of the numbers in each cage must perfectly match its provided target sum.
- Non-Repeating Rule: Numbers cannot repeat within a single cage.

2. Mathematical Facts & Solving Strategies:
- Sum to 21: Since the numbers 1 through 6 add up to 21, every completely filled row, column, and 2x3 subgrid must sum exactly to 21.
- A 6-cell cage must also have a target sum of 21.
- Deduction Tip: For any subset of cells (row, column, or subgrid), if all but one cell are filled, the final cell's value must be 21 minus the sum of the known cells.

3. Reference Material: The Cheat-Sheet
Important Interpretation Rules for the cheat-sheet:
- Combinations are written as sequences of individual digits without separators.
- Each digit in the sequence represents one distinct number in the combination.
- Example: "12" means the combination {{1, 2}}. "135" means the combination {{1, 3, 5}}.
Here is your cheat-sheet for cage combinations:
{cheat_sheet}

4. The Puzzle Instance:
Initial State:
{puzzle}

Cages:
{cages}

5. Output Requirement:
Return ONLY the final solved 6x6 grid. Do not include any explanation or extra text.
"""

In [146]:
class SudokuResult(BaseModel):
    board: List[List[str]] = Field(description="The solved 6x6 Sudoku grid as a 2D list of strings.")

In [147]:
config = types.GenerateContentConfig(
    response_json_schema=SudokuResult.model_json_schema(),
    response_mime_type="application/json"
)

response = client.models.generate_content(
    model=model_LLM,
    contents=prompt.format(puzzle=markdown_table, cages="\n".join(cages_text), cheat_sheet=cheat_sheet), 
    config=config
)
result = SudokuResult.model_validate_json(response.text)

print(grid_to_markdown(result.board))

|  |  |  |  |  |  |
|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 |


# Multi-step prompt

In [148]:
analysis_prompt = """
You are an expert Sudoku reasoning engine. Your current task is strictly limited to ANALYSIS and NOTE-TAKING. 
Do NOT attempt to solve the puzzle completely. Do NOT output a final grid. Do NOT make eager guesses.

Your objective is to carefully scan the current board state, apply standard Sudoku rules along with the specific variant constraints, and generate a logical scratchpad. This scratchpad will be used in the next step to confidently commit digits.

[KNOWLEDGE RECAP]
1. The Grid & Core Rules:
- The grid is 6x6, containing 6 rows and 6 columns.
- Notation: "r1c1" indicates the cell at row 1 and column 1.
- The grid consists of 6 rectangular 2x3 subgrids (2 rows tall, 3 columns wide).
- Standard Sudoku Rule: Each row, each column, and each 2x3 subgrid must contain the numbers 1 through 6 exactly once.
- Killer Sudoku Rule: The grid is divided into "Cages" (contiguous groups of cells). The sum of the numbers in each cage must perfectly match its provided target sum.
- Non-Repeating Rule: Numbers cannot repeat within a single cage.

2. Mathematical Facts & Solving Strategies:
- Sum to 21: Since the numbers 1 through 6 add up to 21, every completely filled row, column, and 2x3 subgrid must sum exactly to 21.
- A 6-cell cage must also have a target sum of 21.
- Deduction Tip: For any subset of cells (row, column, or subgrid), if all but one cell are filled, the final cell's value must be 21 minus the sum of the known cells.

3. Reference Material: The Cheat-Sheet
Important Interpretation Rules for the cheat-sheet:
- Combinations are written as sequences of individual digits without separators.
- Each digit in the sequence represents one distinct number in the combination.
- Example: "12" means the combination {{1, 2}}. "135" means the combination {{1, 3, 5}}.
Here is your cheat-sheet for cage combinations:
{cheat_sheet}

[LAST MOVE LOG]
- Cell Modified: {last_cell} (Value: {last_value})
- AI's Reasoning: "{last_reasoning}"
- Confidence: {last_is_certain}

[CURRENT STATE]
Current board:
{current_board}

Cages:
{cages}

[PREVIOUS ANALYSIS]
{previous_analysis}

INSTRUCTIONS:
1. Scan the board for the most constrained areas (cells, rows, columns, or variant shapes with the fewest empty spaces).
2. Eliminate candidate digits for these empty cells based on the active rules.
3. If a cell is reduced to a single valid digit, explicitly highlight it in the "Deductions" section with irrefutable logical proof.
4. Output your analysis EXACTLY matching the Markdown template below. Do not add conversational filler.

{analysis_template}
"""

In [149]:
fill_prompt = """
You are the Execution Engine of an advanced Sudoku solver. Your task is to evaluate the provided logical analysis and commit EXACTLY ONE valid digit to the board.

[KNOWLEDGE RECAP]
1. The Grid & Core Rules:
- The grid is 6x6, containing 6 rows and 6 columns.
- Notation: "r1c1" indicates the cell at row 1 and column 1.
- The grid consists of 6 rectangular 2x3 subgrids (2 rows tall, 3 columns wide).
- Standard Sudoku Rule: Each row, each column, and each 2x3 subgrid must contain the numbers 1 through 6 exactly once.
- Killer Sudoku Rule: The grid is divided into "Cages" (contiguous groups of cells). The sum of the numbers in each cage must perfectly match its provided target sum.
- Non-Repeating Rule: Numbers cannot repeat within a single cage.

2. Mathematical Facts & Solving Strategies:
- Sum to 21: Since the numbers 1 through 6 add up to 21, every completely filled row, column, and 2x3 subgrid must sum exactly to 21.
- A 6-cell cage must also have a target sum of 21.
- Deduction Tip: For any subset of cells (row, column, or subgrid), if all but one cell are filled, the final cell's value must be 21 minus the sum of the known cells.

3. Reference Material: The Cheat-Sheet
Important Interpretation Rules for the cheat-sheet:
- Combinations are written as sequences of individual digits without separators.
- Each digit in the sequence represents one distinct number in the combination.
- Example: "12" means the combination {{1, 2}}. "135" means the combination {{1, 3, 5}}.
Here is your cheat-sheet for cage combinations:
{cheat_sheet}

[CURRENT STATE]
Current board:
{current_board}

Cages:
{cages}

[LOGICAL ANALYSIS]
{current_analysis}

INSTRUCTIONS:
1. Review the "3. Deductions & Bottlenecks" section of the Logical Analysis carefully.
2. Identify ONE cell where a digit is mathematically forced. You can ONLY modify cells that currently contain 0.
3. If there is NO forced cell, review the "2. Candidate Elimination" section and make your best educated guess for the cell with the fewest candidates.
4. You MUST output your decision for exactly ONE cell. Abstaining is strictly prohibited.
5. Output your response STRICTLY as a valid JSON object.
"""

In [150]:
current_board = deepcopy(puzzle['puzzle'])
previous_analysis = "This is the very first turn. Please generate the initial analysis."
last_cell = "N/A"
last_value = "N/A"
last_reasoning = "N/A"
last_is_certain = "N/A"

analysis_template = None
with open('analysis_template.md', 'r', encoding='utf-8') as file:
    analysis_template = file.read()

def parse_cell(cell_str):
    """
    Parse cell string in format 'rXcY' to (row, col) with 0-indexed coordinates.
    Example: 'r2c3' -> (1, 2)
    """
    # Remove 'r' and 'c', split by 'c'
    parts = cell_str.lower().replace('r', '').split('c')
    row = int(parts[0]) - 1  # Convert to 0-indexed
    col = int(parts[1]) - 1  # Convert to 0-indexed
    return row, col

def update_board(board, cell_str, value):
    """
    Update the board with a new value at the specified cell.
    cell_str: string in format 'rXcY' (e.g., 'r2c3')
    value: integer value to fill
    """
    row, col = parse_cell(cell_str)
    board[row][col] = value
    return board

def validate_current_state(solution, current_board):
    for r in range(6):
        for c in range(6):
            if current_board[r][c] != 0 and current_board[r][c] != solution[r][c]:
                return False
    return True

def is_all_filled(board):
    for row in board:
        for cell in row:
            if cell == 0:
                return False
    return True

execution_log = []
step_count = 0
final_status = "In Progress"
error_type = "None"

In [151]:
class SudokuFillResult(BaseModel):
    reasoning: str = Field(description="Briefly state why this digit is forced, or explain the logic behind your guess.")
    cell: str = Field(description="The cell being filled, in the format 'rXcY'. eg: 'r2c3' which means row 2 column 3 on the board.")
    value: int = Field(description="The digit being filled into the cell. Must be an integer from 1 to 6 for a 6x6 Sudoku.")
    is_certain: bool = Field(description="True if this digit is mathematically forced, False if this is an educated guess based on candidate elimination.")

class SudokuAnalysisResult(BaseModel):
    analysis: str = Field(description="The logical analysis and scratchpad generated by the LLM, formatted according to the specified template.")


In [152]:
def call_llm_for_analysis():
    global previous_analysis

    config = types.GenerateContentConfig(
        temperature=0,
        response_json_schema=SudokuAnalysisResult.model_json_schema(),
        response_mime_type="application/json"
    )

    response = client.models.generate_content(
        model=model_LLM,
        contents=analysis_prompt.format(
            cheat_sheet=cheat_sheet, 
            last_cell=last_cell, 
            last_value=last_value, 
            last_reasoning=last_reasoning, 
            last_is_certain=last_is_certain, 
            current_board=grid_to_markdown(current_board),
            cages="\n".join(cages_text),
            previous_analysis=previous_analysis,
            analysis_template=analysis_template
        ), 
        config=config
    )
    result = SudokuAnalysisResult.model_validate_json(response.text)

    previous_analysis = result.analysis

def call_llm_for_fill():
    global current_board, last_cell, last_value, last_reasoning, last_is_certain
    global execution_log, step_count

    config = types.GenerateContentConfig(
        temperature=0,
        response_json_schema=SudokuFillResult.model_json_schema(),
        response_mime_type="application/json"
    )

    response = client.models.generate_content(
        model=model_LLM,
        contents=fill_prompt.format(
            cheat_sheet=cheat_sheet,
            current_board=grid_to_markdown(current_board),
            cages="\n".join(cages_text),
            current_analysis=previous_analysis
        ), 
        config=config
    )
    result = SudokuFillResult.model_validate_json(response.text)

    current_board = update_board(current_board, result.cell, result.value)
    last_cell = result.cell
    last_value = result.value
    last_reasoning = result.reasoning
    last_is_certain = result.is_certain

    # ← Log phải đặt ở ĐÂY, sau khi result đã có
    step_count += 1
    execution_log.append({
        "step": step_count,
        "analysis_summary": previous_analysis[:300] + "..." if len(previous_analysis) > 300 else previous_analysis,
        "chosen_cell": result.cell,
        "value": result.value,
        "reasoning": result.reasoning,
        "is_certain": result.is_certain,
        "board_state": deepcopy(current_board)
    })


## Loop

In [153]:
while (validate_current_state(puzzle['solution'], current_board) and not is_all_filled(current_board)):
    call_llm_for_analysis()
    call_llm_for_fill()
    print(grid_to_markdown(current_board))

|  |  |  |  |  |  |  |  |  |
|---|---|---|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 5 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
|  |  |  |  |  |  |  |  |  |
|---|---|---|---|---|---|---|---|---|
| 0 | 0 | 0 | 0 | 5 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 2 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
|  |  |  |  |  |  |  |  |  |
|---|---|---|---|---|---|---|---|---|
| 2 | 0 | 0 | 0 | 5 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 2 | 0 | 0 | 0 | 0 |
| 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
|

In [154]:
import os

is_correct = (current_board == puzzle['solution'])
if is_correct:
    final_status = "Success"
    error_type = "None"
elif not is_all_filled(current_board):
    # Dừng sớm vì điền sai nhưng board chưa full
    final_status = "Failed"
    error_type = "Incorrect Solution"
else:
    final_status = "Failed"
    for r in range(6):
        for c in range(6):
            if current_board[r][c] != 0 and current_board[r][c] != puzzle['solution'][r][c]:
                error_type = "Incorrect Solution"
                break

log_data = {
    "puzzle_id": puzzle['id'],
    "difficulty": puzzle['difficulty'],
    "model": model_LLM,
    "total_steps": step_count,
    "final_status": final_status,
    "error_type": error_type,
    "execution_log": execution_log
}

os.makedirs("outputs", exist_ok=True)
out_file = f"outputs/log_puzzle_{puzzle['id']:02d}.json"
with open(out_file, 'w', encoding='utf-8') as f:
    json.dump(log_data, f, ensure_ascii=False, indent=2)

print(f"   Đã lưu log: {out_file}")
print(f"   Puzzle #{puzzle['id']} | Độ khó: {puzzle['difficulty']} | Tổng bước: {step_count}")
print(f"   Kết quả: {final_status} | Loại lỗi: {error_type}")


   Đã lưu log: outputs/log_puzzle_06.json
   Puzzle #6 | Độ khó: hard | Tổng bước: 3
   Kết quả: Failed | Loại lỗi: Incorrect Solution


# Validate result

In [155]:
def validate_board(board, cages):

    # check rows
    for i, row in enumerate(board):

        if sorted(row) != [1,2,3,4,5,6]:
            print(f"Row {i} is invalid: {row}")

    # check columns
    for col in range(6):

        column = []

        for row in range(6):
            column.append(board[row][col])

        if sorted(column) != [1,2,3,4,5,6]:
            print(f"Column {col} is invalid: {column}")

    # check subgrids
    for row_start in range(0, 6, 2):
        for col_start in range(0, 6, 3):

            nums = []

            for r in range(row_start, row_start + 2):
                for c in range(col_start, col_start + 3):
                    nums.append(board[r][c])

            if sorted(nums) != [1,2,3,4,5,6]:

                print(
                    f"Subgrid ({row_start},{col_start}) is invalid: {nums}"
                )

    # check cages
    for idx, cage in enumerate(cages):

        cells = cage["cells"]
        target_sum = cage["sum"]

        total = 0

        for r, c in cells:
            total += board[r][c]

        if total != target_sum:

            print(
                f"Cage {idx + 1} is invalid: "
                f"cells={cells}, "
                f"sum={total}, "
                f"expected={target_sum}"
            )

    print("Validation complete.")

In [156]:
board_int = [[int(cell) for cell in row] for row in result.board]
validate_board(board_int, puzzle['cages'])

Row 0 is invalid: [0, 0, 0, 0, 0, 0]
Row 1 is invalid: [0, 0, 0, 0, 0, 0]
Row 2 is invalid: [0, 0, 0, 0, 0, 0]
Row 3 is invalid: [0, 0, 0, 0, 0, 0]
Row 4 is invalid: [0, 0, 0, 0, 0, 0]
Row 5 is invalid: [0, 0, 0, 0, 0, 0]
Column 0 is invalid: [0, 0, 0, 0, 0, 0]
Column 1 is invalid: [0, 0, 0, 0, 0, 0]
Column 2 is invalid: [0, 0, 0, 0, 0, 0]
Column 3 is invalid: [0, 0, 0, 0, 0, 0]
Column 4 is invalid: [0, 0, 0, 0, 0, 0]
Column 5 is invalid: [0, 0, 0, 0, 0, 0]
Subgrid (0,0) is invalid: [0, 0, 0, 0, 0, 0]
Subgrid (0,3) is invalid: [0, 0, 0, 0, 0, 0]
Subgrid (2,0) is invalid: [0, 0, 0, 0, 0, 0]
Subgrid (2,3) is invalid: [0, 0, 0, 0, 0, 0]
Subgrid (4,0) is invalid: [0, 0, 0, 0, 0, 0]
Subgrid (4,3) is invalid: [0, 0, 0, 0, 0, 0]
Cage 1 is invalid: cells=[[0, 0], [0, 1]], sum=0, expected=6
Cage 2 is invalid: cells=[[0, 2], [0, 3]], sum=0, expected=16
Cage 3 is invalid: cells=[[0, 4], [1, 4]], sum=0, expected=7


IndexError: list index out of range